In [1]:
"""Cricbuzz API exploration.

Every response is cached to data/raw/samples/ on first fetch, so re-running
a cell costs zero API requests.
"""

import json
import os
from pathlib import Path

import requests
from dotenv import load_dotenv


def find_project_root(start: Path) -> Path:
    """Walk upwards until we find the folder containing .git."""
    for candidate in [start, *start.parents]:
        if (candidate / ".git").exists():
            return candidate
    raise RuntimeError("Could not locate the project root")


PROJECT_ROOT = find_project_root(Path.cwd())
SAMPLES_DIR = PROJECT_ROOT / "data" / "raw" / "samples"
SAMPLES_DIR.mkdir(parents=True, exist_ok=True)

load_dotenv(PROJECT_ROOT / ".env")

KEY = os.getenv("RAPIDAPI_KEY")
HOST = os.getenv("RAPIDAPI_HOST")
HEADERS = {"x-rapidapi-key": KEY, "x-rapidapi-host": HOST}

print("project root :", PROJECT_ROOT)
print("samples dir  :", SAMPLES_DIR)
print("host         :", HOST)
print("key          :", "loaded" if KEY else "MISSING")

project root : d:\Tanay Nagpal\Labmentrix Internship\cricbuzz_livestats
samples dir  : d:\Tanay Nagpal\Labmentrix Internship\cricbuzz_livestats\data\raw\samples
host         : cricbuzz-cricket.p.rapidapi.com
key          : loaded


In [2]:
def fetch(path: str, name: str, force: bool = False) -> dict:
    """Fetch an endpoint, caching the response to disk.

    Args:
        path: endpoint path, e.g. "/matches/v1/recent"
        name: filename to cache under, e.g. "matches_recent"
        force: re-fetch even when a cached copy exists

    Returns:
        The parsed JSON response.
    """
    cache_file = SAMPLES_DIR / f"{name}.json"

    if cache_file.exists() and not force:
        print(f"CACHED  {name}.json  -  0 requests used")
        return json.loads(cache_file.read_text(encoding="utf-8"))

    url = f"https://{HOST}{path}"
    print(f"FETCH   {url}")

    response = requests.get(url, headers=HEADERS, timeout=20)
    response.raise_for_status()
    data = response.json()

    cache_file.write_text(json.dumps(data, indent=2), encoding="utf-8")
    size_kb = cache_file.stat().st_size / 1024
    print(f"SAVED   {name}.json  -  {size_kb:.1f} KB  -  1 request used")

    return data


def shape(obj, name: str = "root", depth: int = 0, max_depth: int = 3) -> None:
    """Print the structure of nested JSON without dumping all the values."""
    pad = "  " * depth

    if isinstance(obj, dict):
        print(f"{pad}{name}: dict ({len(obj)} keys)")
        if depth < max_depth:
            for key, value in obj.items():
                shape(value, key, depth + 1, max_depth)

    elif isinstance(obj, list):
        print(f"{pad}{name}: list ({len(obj)} items)")
        if obj and depth < max_depth:
            shape(obj[0], f"{name}[0]", depth + 1, max_depth)

    else:
        preview = str(obj)
        if len(preview) > 40:
            preview = preview[:40] + "..."
        print(f"{pad}{name}: {type(obj).__name__} = {preview}")


print("helpers ready")

helpers ready


In [ ]:
recent = fetch("/matches/v1/recent", "matches_recent")

print()
shape(recent, "recent", max_depth=4)

FETCH   https://cricbuzz-cricket.p.rapidapi.com/matches/v1/recent
SAVED   matches_recent.json  -  195.3 KB  -  1 request used

recent: dict (4 keys)
  typeMatches: list (4 items)
    typeMatches[0]: dict (2 keys)
      matchType: str = International
      seriesMatches: list (6 items)
        seriesMatches[0]: dict (1 keys)
  filters: dict (1 keys)
    matchType: list (4 items)
      matchType[0]: str = International
  appIndex: dict (2 keys)
    seoTitle: str = Live Cricket Score - Scorecard and Match...
    webURL: str = www.cricbuzz.com/live-cricket-scores/
  responseLastUpdated: str = 1789403443


In [4]:
first_type = recent["typeMatches"][0]
print("matchType:", first_type["matchType"])
print("series count:", len(first_type["seriesMatches"]))
print()

first_series = first_type["seriesMatches"][0]
shape(first_series, "seriesMatches[0]", max_depth=6)

matchType: International
series count: 6

seriesMatches[0]: dict (1 keys)
  seriesAdWrapper: dict (3 keys)
    seriesId: int = 12973
    seriesName: str = Afghanistan vs India in India 2026
    matches: list (1 items)
      matches[0]: dict (2 keys)
        matchInfo: dict (19 keys)
          matchId: int = 170103
          seriesId: int = 12973
          seriesName: str = Afghanistan vs India in India 2026
          matchDesc: str = 1st T20I
          matchFormat: str = T20
          startDate: str = 1789308000000
          endDate: str = 1789320600000
          state: str = Complete
          status: str = India won by 7 wkts
          team1: dict (4 keys)
            teamId: int = 96
            teamName: str = Afghanistan
            teamSName: str = AFG
            imageId: int = 776177
          team2: dict (4 keys)
            teamId: int = 2
            teamName: str = India
            teamSName: str = IND
            imageId: int = 776162
          venueInfo: dict (6 keys)
  

In [5]:
from collections import Counter

by_type = Counter()
by_format = Counter()
skipped = 0

for type_block in recent["typeMatches"]:
    match_type = type_block["matchType"]

    for series_block in type_block["seriesMatches"]:
        wrapper = series_block.get("seriesAdWrapper")

        if wrapper is None:
            skipped += 1          # Cricbuzz injects ad entries here
            continue

        for match in wrapper.get("matches", []):
            by_type[match_type] += 1
            by_format[match["matchInfo"]["matchFormat"]] += 1

print("Matches by type:")
for name, count in by_type.most_common():
    print(f"   {name:<18} {count}")

print("\nMatches by format:")
for name, count in by_format.most_common():
    print(f"   {name:<18} {count}")

print(f"\nNon-series entries skipped: {skipped}")

Matches by type:
   Domestic           30
   League             25
   International      18
   Women              16

Matches by format:
   T20                57
   TEST               18
   ODI                14

Non-series entries skipped: 1


In [6]:
match = first_series["seriesAdWrapper"]["matches"][0]
info = match["matchInfo"]

print("matches[0] keys:", list(match.keys()))
print()
print(f"matchInfo — {len(info)} keys:")
for key in info:
    print("   ", key)

matches[0] keys: ['matchInfo', 'matchScore']

matchInfo — 19 keys:
    matchId
    seriesId
    seriesName
    matchDesc
    matchFormat
    startDate
    endDate
    state
    status
    team1
    team2
    venueInfo
    currBatTeamId
    seriesStartDt
    seriesEndDt
    isTimeAnnounced
    stateTitle
    isFantasyEnabled
    hideScoreStatus


In [7]:
from datetime import datetime, timezone


def as_date(value, unit: str) -> str:
    """Interpret an epoch value as seconds or milliseconds."""
    divisor = 1000 if unit == "ms" else 1
    try:
        return str(datetime.fromtimestamp(int(value) / divisor, tz=timezone.utc))
    except (ValueError, OSError, OverflowError):
        return "IMPOSSIBLE - out of range"


for field in ("startDate", "endDate", "seriesStartDt", "seriesEndDt"):
    raw = info[field]
    print(f"{field}: {raw!r}   type={type(raw).__name__}   digits={len(str(raw))}")
    print(f"   as seconds      -> {as_date(raw, 's')}")
    print(f"   as milliseconds -> {as_date(raw, 'ms')}")
    print()

stamp = recent["responseLastUpdated"]
print(f"responseLastUpdated: {stamp!r}   digits={len(str(stamp))}")
print(f"   as seconds      -> {as_date(stamp, 's')}")
print(f"   as milliseconds -> {as_date(stamp, 'ms')}")

startDate: '1789308000000'   type=str   digits=13
   as seconds      -> IMPOSSIBLE - out of range
   as milliseconds -> 2026-09-13 14:00:00+00:00

endDate: '1789320600000'   type=str   digits=13
   as seconds      -> IMPOSSIBLE - out of range
   as milliseconds -> 2026-09-13 17:30:00+00:00

seriesStartDt: '1789257600000'   type=str   digits=13
   as seconds      -> IMPOSSIBLE - out of range
   as milliseconds -> 2026-09-13 00:00:00+00:00

seriesEndDt: '1789776000000'   type=str   digits=13
   as seconds      -> IMPOSSIBLE - out of range
   as milliseconds -> 2026-09-19 00:00:00+00:00

responseLastUpdated: '1789403443'   digits=10
   as seconds      -> 2026-09-14 16:30:43+00:00
   as milliseconds -> 1970-01-21 17:03:23.443000+00:00


In [8]:
def to_utc(value) -> datetime | None:
    """Convert a Cricbuzz epoch value to a UTC datetime.

    Handles both seconds (10 digits) and milliseconds (13 digits), and
    values arriving as either str or int. Returns None for empty values.
    """
    if value in (None, "", 0, "0"):
        return None

    number = int(value)

    # Cricket dates run 1877-2030. In that range seconds are always 10
    # digits and milliseconds always 13, so digit count identifies the unit.
    if len(str(abs(number))) >= 13:
        number = number / 1000

    return datetime.fromtimestamp(number, tz=timezone.utc)


# Test it against every timestamp field we have
for field in ("startDate", "endDate", "seriesStartDt", "seriesEndDt"):
    print(f"{field:<16} -> {to_utc(info[field])}")

print(f"{'responseLastUpdated':<16} -> {to_utc(recent['responseLastUpdated'])}")
print(f"{'None':<16} -> {to_utc(None)}")
print(f"{'empty string':<16} -> {to_utc('')}")

startDate        -> 2026-09-13 14:00:00+00:00
endDate          -> 2026-09-13 17:30:00+00:00
seriesStartDt    -> 2026-09-13 00:00:00+00:00
seriesEndDt      -> 2026-09-19 00:00:00+00:00
responseLastUpdated -> 2026-09-14 16:30:43+00:00
None             -> None
empty string     -> None


In [9]:
match_id = info["matchId"]
print("fetching scorecard for match", match_id, "-", info["matchDesc"])
print()

scorecard = fetch(f"/mcenter/v1/{match_id}/hscard", f"scorecard_{match_id}")

print()
shape(scorecard, "scorecard", max_depth=3)

fetching scorecard for match 170103 - 1st T20I

FETCH   https://cricbuzz-cricket.p.rapidapi.com/mcenter/v1/170103/hscard
SAVED   scorecard_170103.json  -  38.1 KB  -  1 request used

scorecard: dict (5 keys)
  scorecard: list (2 items)
    scorecard[0]: dict (16 keys)
      inningsid: int = 1
      batsman: list (11 items)
      bowler: list (7 items)
      fow: dict (1 keys)
      extras: dict (6 keys)
      score: int = 156
      wickets: int = 8
      overs: float = 20.0
      runrate: float = 7.8
      batteamname: str = Afghanistan
      batteamsname: str = AFG
      isdeclared: bool = False
      isfollowon: bool = False
      ballnbr: int = 200
      rpb: float = 0.78
      partnership: dict (1 keys)
  ismatchcomplete: bool = True
  appindex: dict (2 keys)
    seotitle: str = Cricket scorecard - AFG vs IND 1st T20I,...
    weburl: str = http://www.cricbuzz.com/live-cricket-sco...
  status: str = India won by 7 wkts
  responselastupdated: int = 1789405443


In [10]:
innings = scorecard["scorecard"][0]

print(f"Innings {innings['inningsid']}: {innings['batteamname']} "
      f"{innings['score']}/{innings['wickets']} in {innings['overs']} overs")
print()

first_batter = innings["batsman"][0]
print(f"batsman[0] — {len(first_batter)} keys:")
for key, value in first_batter.items():
    print(f"   {key:<18} {value!r}")

Innings 1: Afghanistan 156/8 in 20.0 overs

batsman[0] — 22 keys:
   id                 13213
   balls              2
   runs               0
   fours              0
   sixes              0
   strkrate           '0'
   name               'Rahmanullah Gurbaz'
   nickname           'Rahmanullah Gurbaz'
   iscaptain          False
   iskeeper           True
   outdec             'c Abhishek Sharma b Jasprit Bumrah'
   videotype          ''
   videourl           ''
   videoid            0
   planid             0
   imageid            0
   premiumvideourl    ''
   iscbplusfree       False
   ispremiumfree      False
   inmatchchange      ''
   isoverseas         False
   playingxichange    ''


In [11]:
first_bowler = innings["bowler"][0]

print(f"bowler[0] — {len(first_bowler)} keys:")
for key, value in first_bowler.items():
    print(f"   {key:<18} {value!r}")


bowler[0] — 23 keys:
   id                 9311
   overs              '4'
   maidens            0
   wickets            1
   runs               26
   economy            '6.5'
   name               'Jasprit Bumrah'
   nickname           'Jasprit Bumrah'
   iscaptain          False
   iskeeper           False
   videotype          ''
   videourl           ''
   videoid            0
   dots               0
   balls              40
   rpb                0.65
   planid             0
   imageid            0
   premiumvideourl    ''
   ispremiumfree      False
   inmatchchange      ''
   isoverseas         False
   playingxichange    ''


In [12]:
print(f"{'bowler':<22}{'overs':>7}{'balls':>8}{'overs×10':>10}{'match?':>8}")
print("-" * 55)

for bowler in innings["bowler"]:
    overs = float(bowler["overs"])
    reported = bowler["balls"]
    expected = overs * 10
    print(f"{bowler['name']:<22}{overs:>7}{reported:>8}{expected:>10.0f}"
          f"{'YES' if reported == expected else 'NO':>8}")

bowler                  overs   balls  overs×10  match?
-------------------------------------------------------
Jasprit Bumrah            4.0      40        40     YES
Arshdeep Singh            4.0      40        40     YES
Axar Patel                4.0      40        40     YES
Varun Chakaravarthy       4.0      40        40     YES
Nitish Kumar Reddy        2.0      20        20     YES
Abhishek Sharma           1.0      10        10     YES
Shivam Dube               1.0      10        10     YES


In [13]:
def overs_to_balls(overs) -> int:
    """Convert cricket overs notation to a true ball count.

    In cricket '3.4' means 3 overs and 4 balls = 22 balls, not 3.4 x 6.
    """
    overs = float(overs)
    whole = int(overs)
    part = round((overs - whole) * 10)
    return whole * 6 + part


for value in ("4", "3.4", "0.3", "10.5", "19.2", "20"):
    print(f"{value:>6} overs  ->  {overs_to_balls(value):>4} balls")

print("\nLooking for a partial over in this match...")
found = False

for inn in scorecard["scorecard"]:
    for b in inn["bowler"]:
        overs = float(b["overs"])
        if overs != int(overs):
            print(f"   {b['name']:<22} overs={b['overs']:<6} "
                  f"api_balls={b['balls']:<5} real_balls={overs_to_balls(overs)}")
            found = True

if not found:
    print("   none - every bowler in this match bowled whole overs")

     4 overs  ->    24 balls
   3.4 overs  ->    22 balls
   0.3 overs  ->     3 balls
  10.5 overs  ->    65 balls
  19.2 overs  ->   116 balls
    20 overs  ->   120 balls

Looking for a partial over in this match...
   Rashid Khan            overs=3.4    api_balls=34    real_balls=22


In [14]:
upcoming = fetch("/matches/v1/upcoming", "matches_upcoming")

print()
shape(upcoming, "upcoming", max_depth=3)

FETCH   https://cricbuzz-cricket.p.rapidapi.com/matches/v1/upcoming
SAVED   matches_upcoming.json  -  29.8 KB  -  1 request used

upcoming: dict (4 keys)
  typeMatches: list (4 items)
    typeMatches[0]: dict (2 keys)
      matchType: str = International
      seriesMatches: list (5 items)
  filters: dict (1 keys)
    matchType: list (4 items)
      matchType[0]: str = International
  appIndex: dict (2 keys)
    seoTitle: str = Live Cricket Score - Scorecard and Match...
    webURL: str = www.cricbuzz.com/live-cricket-scores/
  responseLastUpdated: str = 1789409562


In [15]:
# Skip ad entries when finding the first real series
wrapper = None
for block in upcoming["typeMatches"][0]["seriesMatches"]:
    if "seriesAdWrapper" in block:
        wrapper = block["seriesAdWrapper"]
        break

upcoming_match = wrapper["matches"][0]
info_up = upcoming_match["matchInfo"]

print("series :", wrapper["seriesName"])
print("match  :", info_up["matchDesc"], "-", info_up["matchFormat"])
print("keys on matches[0]:", list(upcoming_match.keys()))
print()
print("state  :", info_up["state"])
print("status :", info_up["status"])
print("starts :", to_utc(info_up["startDate"]))
print()

print("--- field comparison with the COMPLETED match ---")
print("only in completed:", set(info.keys()) - set(info_up.keys()))
print("only in upcoming :", set(info_up.keys()) - set(info.keys()))

series : Afghanistan vs India in India 2026
match  : 2nd T20I - T20
keys on matches[0]: ['matchInfo']

state  : Preview
status : Match starts at Sep 15, 14:00 GMT
starts : 2026-09-15 14:00:00+00:00

--- field comparison with the COMPLETED match ---
only in completed: {'currBatTeamId'}
only in upcoming : set()


In [16]:
live = fetch("/matches/v1/live", "matches_live")

total = 0
for type_block in live.get("typeMatches", []):
    for series_block in type_block["seriesMatches"]:
        wrapper = series_block.get("seriesAdWrapper")
        if wrapper:
            total += len(wrapper.get("matches", []))

print(f"\nlive matches right now: {total}")
print()
shape(live, "live", max_depth=2)

FETCH   https://cricbuzz-cricket.p.rapidapi.com/matches/v1/live


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [17]:
url = f"https://{HOST}/matches/v1/live"
response = requests.get(url, headers=HEADERS, timeout=20)

print("status code  :", response.status_code)
print("content-type :", response.headers.get("content-type"))
print("body length  :", len(response.content), "bytes")
print("first 300    :", repr(response.text[:300]))

status code  : 204
content-type : None
body length  : 0 bytes
first 300    : ''


In [18]:
def fetch(path: str, name: str, force: bool = False) -> dict:
    """Fetch an endpoint, caching the response to disk.

    Returns {} when the server sends 204 No Content or an empty body.
    Empty responses are not cached - "nothing live right now" is a
    temporary state, not a fact worth remembering.
    """
    cache_file = SAMPLES_DIR / f"{name}.json"

    if cache_file.exists() and not force:
        print(f"CACHED  {name}.json  -  0 requests used")
        return json.loads(cache_file.read_text(encoding="utf-8"))

    url = f"https://{HOST}{path}"
    print(f"FETCH   {url}")

    response = requests.get(url, headers=HEADERS, timeout=20)
    print(f"        HTTP {response.status_code}, {len(response.content)} bytes")

    if response.status_code >= 400:
        print(f"ERROR   {response.status_code}: {response.text[:200]}")
        return {}

    if response.status_code == 204 or not response.content.strip():
        print("EMPTY   server sent no content - nothing to cache")
        return {}

    try:
        data = response.json()
    except ValueError:
        print(f"BADJSON response was not valid JSON: {response.text[:200]!r}")
        return {}

    cache_file.write_text(json.dumps(data, indent=2), encoding="utf-8")
    size_kb = cache_file.stat().st_size / 1024
    print(f"SAVED   {name}.json  -  {size_kb:.1f} KB  -  1 request used")

    return data


print("fetch() upgraded")

fetch() upgraded


In [19]:
topstats = fetch("/stats/v1/topstats", "topstats")

print()
shape(topstats, "topstats", max_depth=3)

FETCH   https://cricbuzz-cricket.p.rapidapi.com/stats/v1/topstats
        HTTP 200, 1164 bytes
SAVED   topstats.json  -  2.1 KB  -  1 request used

topstats: dict (1 keys)
  statsTypesList: list (2 items)
    statsTypesList[0]: dict (2 keys)
      types: list (9 items)
      category: str = Batting


In [20]:
for block in topstats["statsTypesList"]:
    print(f"\n{block['category'].upper()}")
    for stat_type in block["types"]:
        print("   ", stat_type)


BATTING
    {'value': 'mostRuns', 'header': 'Most Runs', 'category': 'Batting'}
    {'value': 'highestScore', 'header': 'Highest Scores', 'category': 'Batting'}
    {'value': 'highestAvg', 'header': 'Best Batting Average', 'category': 'Batting'}
    {'value': 'highestSr', 'header': 'Best Batting Strike Rate', 'category': 'Batting'}
    {'value': 'mostHundreds', 'header': 'Most Hundreds', 'category': 'Batting'}
    {'value': 'mostFifties', 'header': 'Most Fifties', 'category': 'Batting'}
    {'value': 'mostFours', 'header': 'Most Fours', 'category': 'Batting'}
    {'value': 'mostSixes', 'header': 'Most Sixes', 'category': 'Batting'}
    {'value': 'mostNineties', 'header': 'Most Nineties', 'category': 'Batting'}

BOWLING
    {'value': 'mostWickets', 'header': 'Most Wickets', 'category': 'Bowling'}
    {'value': 'lowestAvg', 'header': 'Best Bowling Average', 'category': 'Bowling'}
    {'value': 'bestBowlingInnings', 'header': 'Best Bowling ', 'category': 'Bowling'}
    {'value': 'mostFiv